# Document Question Answering System (RAG)

**Week 7 Assignment**

This notebook builds a complete Retrieval-Augmented Generation (RAG) pipeline from scratch.
The idea is simple — instead of asking a language model to answer from memory, we first
fetch the most relevant parts of our own document, then pass those chunks as context


### 7-Stage Architecture

| Stage | Description |
|---|---|
| 1 | Document Ingestion
| 2 | Text Chunking
| 3 | Embedding Creation
| 4 | Vector Database
| 5 | Query Processing
| 6 | Context Retrieval
| 7 | Answer Generation

## Step 1 — Install Required Libraries


In [3]:
!pip install -q sentence-transformers faiss-cpu transformers pypdf scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 34.8 MB/s eta 0:00:00


## Step 2 — Import Libraries

In [4]:
import os
import textwrap
import numpy as np
import faiss
from typing import List, Tuple

from sentence_transformers import SentenceTransformer
from transformers import T5ForConditionalGeneration, T5Tokenizer

print('All libraries imported successfully!')

All libraries imported successfully!


## Stage 1 — Document Ingestion

Load raw text from a `.txt` or `.pdf` file.


In [5]:
# create a sample document about AI topics to test the pipeline
sample_text = """Artificial Intelligence (AI) refers to the simulation of human intelligence in machines.
These machines are programmed to think like humans and mimic their actions.
The term may also be applied to any machine that exhibits traits associated with a human mind
such as learning and problem-solving.

Machine Learning is a subset of AI that gives systems the ability to learn and improve from
experience without being explicitly programmed. It focuses on developing programs that can
access data and use it to learn for themselves.

Deep Learning is part of a broader family of machine learning methods based on artificial
neural networks with representation learning. Learning can be supervised, semi-supervised
or unsupervised.

Natural Language Processing (NLP) is a subfield of AI concerned with the interactions
between computers and human language. It involves processing and analyzing large amounts
of natural language data.

A Transformer is a deep learning model that uses the self-attention mechanism to weigh
the significance of each part of the input data. It is used primarily in NLP and computer
vision tasks.

BERT (Bidirectional Encoder Representations from Transformers) was created and published
in 2018 by researchers at Google. It is pre-trained on a large corpus of text and can be
fine-tuned for various downstream NLP tasks.

GPT (Generative Pre-trained Transformer) is a large language model developed by OpenAI.
It uses deep learning to produce human-like text for tasks like translation, summarization,
and question answering.

Retrieval-Augmented Generation (RAG) combines retrieval-based methods with generative
language models. Instead of relying only on the model's internal knowledge, RAG retrieves
relevant documents from an external knowledge base and uses them as context to generate
more accurate and factually grounded answers.

Vector databases store data as high-dimensional vectors and allow efficient similarity
search. Popular vector databases include FAISS, Pinecone, Weaviate, and ChromaDB.
They are a core component of modern RAG systems.

Embeddings are numerical representations of text in a continuous vector space.
Semantically similar texts have embeddings that are close together in this space.
The sentence-transformers library is widely used for generating high-quality text embeddings.
"""

with open('sample_document.txt', 'w') as f:
    f.write(sample_text)

print('Sample document created.')
print(f'Total characters: {len(sample_text)}')

Sample document created.
Total characters: 2336


In [6]:
def load_document(file_path: str) -> str:
    """
    Stage 1: Document Ingestion
    Reads a .txt or .pdf file and returns its full text content.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f'File not found: {file_path}')

    ext = os.path.splitext(file_path)[1].lower()

    if ext == '.txt':
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()

    elif ext == '.pdf':

        from pypdf import PdfReader
        reader = PdfReader(file_path)
        return '\n'.join(page.extract_text() or '' for page in reader.pages)

    else:
        raise ValueError(f"Unsupported format '{ext}'. Use .txt or .pdf")


# load the sample
raw_text = load_document('sample_document.txt')

print(f'Document loaded successfully!')
print(f'Total characters: {len(raw_text)}')
print(f'\nFirst 300 characters preview:')
print(raw_text[:300])

Document loaded successfully!
Total characters: 2336

First 300 characters preview:
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines.
These machines are programmed to think like humans and mimic their actions.
The term may also be applied to any machine that exhibits traits associated with a human mind
such as learning and problem-solving.

Ma


## Stage 2 — Text Chunking

Documents can be very long — we can't embed or pass an entire book to a model at once.
So we split the text into smaller **overlapping** windows (chunks).

The **overlap** is important: if a chunk boundary falls mid-sentence, that information
would be split across two chunks.

In [7]:
def chunk_text(text: str, chunk_size: int = 80, overlap: int = 15) -> List[str]:
    """
    Stage 2: Text Chunking
    Splits text into overlapping chunks of ~chunk_size words.

    Args:
        text: the full document text
        chunk_size: number of words per chunk
        overlap: words shared between consecutive chunks

    Returns:
        List of text chunk strings
    """
    words = text.split()
    if not words:
        return []

    chunks = []
    start = 0
    step = max(chunk_size - overlap, 1)

    while start < len(words):
        end = start + chunk_size
        chunks.append(' '.join(words[start:end]))
        if end >= len(words):
            break
        start += step

    return chunks


chunks = chunk_text(raw_text, chunk_size=80, overlap=15)

print(f'Total chunks created: {len(chunks)}')
print('\n--- Sample Chunks ---')
for i, chunk in enumerate(chunks[:3]):
    print(f'\nChunk {i+1}:')
    print(textwrap.fill(chunk, width=80))
    print('-' * 60)

Total chunks created: 5

--- Sample Chunks ---

Chunk 1:
Artificial Intelligence (AI) refers to the simulation of human intelligence in
machines. These machines are programmed to think like humans and mimic their
actions. The term may also be applied to any machine that exhibits traits
associated with a human mind such as learning and problem-solving. Machine
Learning is a subset of AI that gives systems the ability to learn and improve
from experience without being explicitly programmed. It focuses on developing
programs that can access data and use it
------------------------------------------------------------

Chunk 2:
being explicitly programmed. It focuses on developing programs that can access
data and use it to learn for themselves. Deep Learning is part of a broader
family of machine learning methods based on artificial neural networks with
representation learning. Learning can be supervised, semi-supervised or
unsupervised. Natural Language Processing (NLP) is a subfield of A

## Stage 3 — Embedding Creation

Each chunk is converted into a numeric vector (embedding) using `sentence-transformers`.
The model `all-MiniLM-L6-v2` maps text into a 384-dimensional vector space.

Chunks that talk about similar things will have vectors that are **close together**


In [8]:
print('Loading embedding model (all-MiniLM-L6-v2)...')
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

print('Generating embeddings for all chunks...')
chunk_embeddings = embed_model.encode(
    chunks,
    normalize_embeddings=True,   # normalize so inner product = cosine similarity
    show_progress_bar=True
)
chunk_embeddings = np.array(chunk_embeddings, dtype='float32')

print(f'\nEmbedding matrix shape: {chunk_embeddings.shape}')
print(f'({chunk_embeddings.shape[0]} chunks × {chunk_embeddings.shape[1]} dimensions)')

Loading embedding model (all-MiniLM-L6-v2)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings for all chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding matrix shape: (5, 384)
(5 chunks × 384 dimensions)


## Stage 4 — Vector Database (FAISS)

We store all chunk embeddings in a FAISS index for fast similarity search.
FAISS (Facebook AI Similarity Search) is the same category of tool as Pinecone,
Weaviate, and ChromaDB — just running locally in memory.


In [9]:
dimension = chunk_embeddings.shape[1]

# IndexFlatIP = exact inner product search
# for normalized vectors this equals cosine similarity
faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(chunk_embeddings)

print(f'FAISS index built successfully!')
print(f'Vectors stored: {faiss_index.ntotal}')
print(f'Vector dimension: {dimension}')

FAISS index built successfully!
Vectors stored: 5
Vector dimension: 384


## Stages 5 & 6 — Query Processing + Context Retrieval

When a user asks a question:
1. The question is embedded using the **same model** as the chunks
2. FAISS searches for the top-k chunks whose vectors are closest to the query
3. Those chunks are returned as context for the language model

In [10]:
def retrieve(query: str, top_k: int = 3) -> List[Tuple[str, float]]:
    """
    Stages 5-6: Query Processing + Context Retrieval
    Embeds the query and returns the top_k most relevant chunks.

    Args:
        query: the user's question
        top_k: number of chunks to retrieve

    Returns:
        List of (chunk_text, similarity_score) tuples
    """
    query_vec = embed_model.encode([query], normalize_embeddings=True)
    query_vec = np.array(query_vec, dtype='float32')

    scores, indices = faiss_index.search(query_vec, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx != -1:
            results.append((chunks[idx], float(score)))
    return results


# quick retrieval test
test_query = 'What is RAG?'
test_results = retrieve(test_query, top_k=3)

print(f'Query: "{test_query}"')
print('\nTop retrieved chunks:')
for i, (chunk, score) in enumerate(test_results, 1):
    print(f'\n[{i}] Similarity score: {score:.4f}')
    print(textwrap.fill(chunk, width=80))

Query: "What is RAG?"

Top retrieved chunks:

[1] Similarity score: 0.2647
corpus of text and can be fine-tuned for various downstream NLP tasks. GPT
(Generative Pre-trained Transformer) is a large language model developed by
OpenAI. It uses deep learning to produce human-like text for tasks like
translation, summarization, and question answering. Retrieval-Augmented
Generation (RAG) combines retrieval-based methods with generative language
models. Instead of relying only on the model's internal knowledge, RAG retrieves
relevant documents from an external knowledge base and uses them as context to
generate more accurate and factually grounded

[2] Similarity score: 0.2630
external knowledge base and uses them as context to generate more accurate and
factually grounded answers. Vector databases store data as high-dimensional
vectors and allow efficient similarity search. Popular vector databases include
FAISS, Pinecone, Weaviate, and ChromaDB. They are a core component of modern RAG
sys

## Stage 7 — Answer Generation

We use `google/flan-t5-base` — a free, open-source instruction-following model from Google.
It's a **seq2seq** (encoder-decoder) model, so we load it directly using
`T5ForConditionalGeneration` and `T5Tokenizer` rather than the pipeline API
(which dropped seq2seq support in newer versions of transformers).

In [11]:
print('Loading flan-t5-base model and tokenizer...')
print('First run downloads ~1GB — cached after that.')

# load tokenizer and model directly — avoids pipeline API version issues
tokenizer = T5Tokenizer.from_pretrained('google/flan-t5-base')
model = T5ForConditionalGeneration.from_pretrained('google/flan-t5-base')

print('\nModel loaded successfully!')


PROMPT_TEMPLATE = """Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't have enough information in the document."

Context:
{context}

Question: {question}

Answer:"""


def generate_answer(question: str, context_chunks: List[Tuple[str, float]]) -> str:
    """
    Stage 7: Answer Generation
    Builds an augmented prompt from the retrieved chunks and
    generates an answer using flan-t5.

    Args:
        question: the user's question
        context_chunks: list of (chunk_text, score) from the retriever

    Returns:
        Generated answer string
    """
    context = '\n\n'.join(chunk for chunk, _ in context_chunks)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)

    # tokenize the prompt
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        max_length=512,
        truncation=True
    )

    # generate the answer
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()


print('Generator function ready!')

Loading flan-t5-base model and tokenizer...
First run downloads ~1GB — cached after that.


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


Model loaded successfully!
Generator function ready!


## Full RAG Pipeline

In [12]:
def ask(question: str, top_k: int = 3):
    """
    Full RAG pipeline:
      - Retrieves the most relevant chunks from the document
      - Augments the prompt with that context
      - Generates a final answer using flan-t5
    """
    # Stages 5-6: retrieve
    context_chunks = retrieve(question, top_k=top_k)

    # Stage 7: generate
    answer = generate_answer(question, context_chunks)

    print('=' * 70)
    print(f'QUESTION: {question}')
    print('-' * 70)
    print('RETRIEVED CONTEXT:')
    for i, (chunk, score) in enumerate(context_chunks, 1):
        print(f'  [{i}] (score={score:.3f})')
        print(textwrap.fill(chunk, width=65, initial_indent='       ',
                            subsequent_indent='       '))
    print('-' * 70)
    print(f'ANSWER: {answer}')
    print('=' * 70)
    return answer


print("Pipeline ready! Use ask('your question') to query the document.")

Pipeline ready! Use ask('your question') to query the document.


## Step 10 — Test the RAG System

Let's run several questions

In [13]:
# Question 1
_ = ask('What is Retrieval-Augmented Generation?')

QUESTION: What is Retrieval-Augmented Generation?
----------------------------------------------------------------------
RETRIEVED CONTEXT:
  [1] (score=0.487)
       corpus of text and can be fine-tuned for various
       downstream NLP tasks. GPT (Generative Pre-trained
       Transformer) is a large language model developed by
       OpenAI. It uses deep learning to produce human-like text
       for tasks like translation, summarization, and question
       answering. Retrieval-Augmented Generation (RAG) combines
       retrieval-based methods with generative language models.
       Instead of relying only on the model's internal knowledge,
       RAG retrieves relevant documents from an external
       knowledge base and uses them as context to generate more
       accurate and factually grounded
  [2] (score=0.250)
       involves processing and analyzing large amounts of natural
       language data. A Transformer is a deep learning model that
       uses the self-attention mech

In [14]:
# Question 2
_ = ask('What are vector databases and give some examples?')

QUESTION: What are vector databases and give some examples?
----------------------------------------------------------------------
RETRIEVED CONTEXT:
  [1] (score=0.544)
       external knowledge base and uses them as context to
       generate more accurate and factually grounded answers.
       Vector databases store data as high-dimensional vectors
       and allow efficient similarity search. Popular vector
       databases include FAISS, Pinecone, Weaviate, and ChromaDB.
       They are a core component of modern RAG systems.
       Embeddings are numerical representations of text in a
       continuous vector space. Semantically similar texts have
       embeddings that are close together in this space. The
       sentence-transformers library is widely used for
       generating high-quality text embeddings.
  [2] (score=0.235)
       Artificial Intelligence (AI) refers to the simulation of
       human intelligence in machines. These machines are
       programmed to think like

In [15]:
# Question 3
_ = ask('What is BERT and who developed it?')

QUESTION: What is BERT and who developed it?
----------------------------------------------------------------------
RETRIEVED CONTEXT:
  [1] (score=0.427)
       involves processing and analyzing large amounts of natural
       language data. A Transformer is a deep learning model that
       uses the self-attention mechanism to weigh the
       significance of each part of the input data. It is used
       primarily in NLP and computer vision tasks. BERT
       (Bidirectional Encoder Representations from Transformers)
       was created and published in 2018 by researchers at
       Google. It is pre-trained on a large corpus of text and
       can be fine-tuned for various downstream NLP tasks. GPT
       (Generative Pre-trained
  [2] (score=0.276)
       being explicitly programmed. It focuses on developing
       programs that can access data and use it to learn for
       themselves. Deep Learning is part of a broader family of
       machine learning methods based on artificial n

In [16]:
# Question 4
_ = ask('What is the difference between Machine Learning and Deep Learning?')

QUESTION: What is the difference between Machine Learning and Deep Learning?
----------------------------------------------------------------------
RETRIEVED CONTEXT:
  [1] (score=0.554)
       being explicitly programmed. It focuses on developing
       programs that can access data and use it to learn for
       themselves. Deep Learning is part of a broader family of
       machine learning methods based on artificial neural
       networks with representation learning. Learning can be
       supervised, semi-supervised or unsupervised. Natural
       Language Processing (NLP) is a subfield of AI concerned
       with the interactions between computers and human
       language. It involves processing and analyzing large
       amounts of natural language data. A Transformer is a deep
  [2] (score=0.515)
       Artificial Intelligence (AI) refers to the simulation of
       human intelligence in machines. These machines are
       programmed to think like humans and mimic their acti

In [17]:
# Question 5
_ = ask('What are embeddings and how are they used?')

QUESTION: What are embeddings and how are they used?
----------------------------------------------------------------------
RETRIEVED CONTEXT:
  [1] (score=0.579)
       external knowledge base and uses them as context to
       generate more accurate and factually grounded answers.
       Vector databases store data as high-dimensional vectors
       and allow efficient similarity search. Popular vector
       databases include FAISS, Pinecone, Weaviate, and ChromaDB.
       They are a core component of modern RAG systems.
       Embeddings are numerical representations of text in a
       continuous vector space. Semantically similar texts have
       embeddings that are close together in this space. The
       sentence-transformers library is widely used for
       generating high-quality text embeddings.
  [2] (score=0.316)
       being explicitly programmed. It focuses on developing
       programs that can access data and use it to learn for
       themselves. Deep Learning is pa

## Step 11 — Interactive Query Mode

Keep asking questions until you type `exit`.

In [18]:
print("Interactive RAG Q&A — type your question below (or 'exit' to stop)\n")

while True:
    user_q = input('Q: ').strip()
    if user_q.lower() in ('exit', 'quit', 'q'):
        print('Exiting Q&A mode.')
        break
    if not user_q:
        print('Please enter a valid question.')
        continue
    ask(user_q)

Interactive RAG Q&A — type your question below (or 'exit' to stop)

Q: q
Exiting Q&A mode.
